# Bulls and cows - can we solve it using entropy

In [ ]:
%load_ext autoreload
%autoreload 2

import random
from collections import Counter
from collections.abc import Callable
from dataclasses import dataclass

import numpy as np
import plotly.express as px

from playground_package.bulls_and_cows.code.bulls_and_cows_game import (
    BullsAndCowsGame,
    GameState,
)
from playground_package.bulls_and_cows.code.feedback_table import FeedbackTable
from playground_package.bulls_and_cows.code.strategies import (
    BaselineStrategy,
    EntropyStrategy,
    IterateThroughAllPossibilitiesStrategy,
    PlayStrategy,
)


In [ ]:
# import importlib
# import playground_package.bulls_and_cows.example as example

# importlib.reload(example)
# Sample = example.Sample

In [ ]:
test_secret = "1233"
game = BullsAndCowsGame(external_secret = test_secret)

def assert_guess(actual, expected):
    assert actual == expected, f"expected {expected}, got {actual}"


assert_guess(game.make_guess("1332"), (2, 2))
assert_guess(game.make_guess("5111"), (0, 1))
assert_guess(game.make_guess("1331"), (2, 1))
assert_guess(game.make_guess("3339"), (1, 1))
assert_guess(game.make_guess("3333"), (2, 0))
assert_guess(game.make_guess("5678"), (0, 0))
assert_guess(game.make_guess("1256"), (2, 0))
assert_guess(game.make_guess("3321"), (0, 4))
assert_guess(game.make_guess("3312"), (0, 4))
assert_guess(game.make_guess("3789"), (0, 1))
assert_guess(game.make_guess("3389"), (0, 2))
assert_guess(game.make_guess("8911"), (0, 1))
assert_guess(BullsAndCowsGame.detect_bulls_and_cows("1122", "2211"), (0, 4))
assert_guess(game.make_guess(test_secret), (4, 0))
assert_guess(game.game_state, GameState.WON)

In [ ]:
play_game = False

if play_game:
    game = BullsAndCowsGame()

    while(True):
        guess = input("What is your guess?")
        if guess == 'q':
            print(f"The number was: {game.secret}")
            break
        bulls, cows = game.make_guess(guess)
        if game.game_state == GameState.WON:
            print(f"You won with: {guess} in {game.guess_count}")
            break
        else:
            print(f"Turn: {game.guess_count} - guess: {guess} - bulls: {bulls}, cows: {cows}")



# Play the game

In [ ]:
@dataclass(frozen=True, slots=True)
class GameResult:
    game_state: GameState
    number_of_turns: int


@dataclass(frozen=True, slots=True)
class FacilitatorRunSummary:
    total_games: int
    won_games: int
    average_turns: float
    all_games_results: list[GameResult]


def plot_results_hist(result: FacilitatorRunSummary):
    turns = [x.number_of_turns for x in result.all_games_results]

    fig = px.histogram(
        x=turns,
        nbins=max(turns) - min(turns) + 1,
        range_x=[min(turns), max(turns)],
        title="Distribution of turns per game",
        labels={"x": "Number of turns", "y": "Games"},
    )

    fig.update_layout(
        bargap=0.1,
        xaxis={"dtick": 1},
        template="plotly_white"
    )

    fig.show()

class GameFacilitator:
    def __init__(
            self,
            game_factory: Callable[[], BullsAndCowsGame],
            strategy_factory: Callable[[], PlayStrategy]):

        self.game_factory = game_factory
        self.strategy_factory = strategy_factory


    def play_games(self, count: int, turn_frequency_log: int = 100) -> FacilitatorRunSummary:
        game_results: list[GameResult] = []

        for i in range(count):
            game = self.game_factory()
            strategy = self.strategy_factory()

            if i % turn_frequency_log == 0:
                print("Game: ", i)

            turn_counter = 0
            while True:
                turn_counter += 1

                guess = strategy.generate_guess()
                bulls, cows = game.make_guess(guess)
                strategy.record_turn_feedback(bulls, cows)

                if game.game_state != GameState.IN_PROGRESS:
                    result = GameResult(game.game_state, turn_counter)
                    game_results.append(result)
                    break

        run_result = FacilitatorRunSummary(
            total_games=count,
            won_games = sum(1 for x in game_results if x.game_state == GameState.WON),
            average_turns = sum(x.number_of_turns for x in game_results) / len(game_results),
            all_games_results = game_results
        )

        return run_result


In [ ]:
game_factory = lambda: BullsAndCowsGame()
strategy_factory = lambda: IterateThroughAllPossibilitiesStrategy()

facilitator = GameFacilitator(game_factory, strategy_factory)

facilitator.play_games(10)


# Solve it choosing next turn based on information gain

In [ ]:
sum(1 for x in range(1000, 10000))

In [ ]:
# I would like to solve the game of Bulls And Cows using information gain analysis
# information that an observation gives: i = -log(p), where p is the probability of the observation
# entropy: e[i] = sum(-p*log(p)) - or expected value of the information, this is a measurement of how much uncertainty there is in the entire set

# So what is the initial entropy/uncertainty for Bulls and Cows?
possibilities = 9000
p = 1 / possibilities

pos_array = np.full(possibilities, p)

entropy_initial = sum(-p * np.log2(pos_array))
print("initial entropy:", entropy_initial)
# initial entropy: 13.135709286102902
# What does that mean? That you should be able to encode every initial state of the game in 13.13 binary questions


In [ ]:
# Algo:
# OUTER LOOP: for each possibility [1000 to 9999],
# counter = Counter
# INNER LOOP: for each other possibility [1000 to 9999]
#   - find the number of bulls and cows.
#   - counter[(bulls, cows)]
# compute -sum( (count/total) * log2(count/total) )
# Example:
    # outer: X [1000, 9999]
    # inner: 1000...Y....9999
    # (bulls, cows) = detect_bulls_cows(inner, outer)
    # counter[(bulls, cows)] += 1
    # E[I] for Y = sum(-p*)*log2(p)


def calc_entropy(guess: str, possibilities: list[str]) -> float:
    counter = Counter()

    for secret in possibilities:
        bulls_and_cows = BullsAndCowsGame.detect_bulls_and_cows(str(secret), str(guess))
        counter[bulls_and_cows] += 1

    total = counter.total()
    entropy = sum(-(count / total) * np.log2(count / total) for count in counter.values())

    return entropy


possibilities = range(1000, 10000)
sorted([
    (calc_entropy(str(guess), possibilities), guess)
    for guess in [1000, 1111, 1234, 5678, 1122, 9999, 1223, 1222]
], reverse=True)

In [ ]:
# Feedback for every (guess, secret) pair, built with numpy broadcasting.
# See feedback_table.py.
feedback_table = FeedbackTable()
feedback_table.feedback("1122", "1234")  # (bulls, cows)

In [ ]:
# Entropy of every number as a first guess, against all 9000 secrets
entropy_9k = feedback_table.entropies()

In [ ]:
guesses = ["1000", "1111", "1234", "5678", "1122", "9999", "1223", "1222"]
sorted(zip(feedback_table.entropies(guesses), guesses), reverse=True)

In [ ]:
candidates, max_val = feedback_table.best_opening_guesses
guess = random.choice(candidates)
guess

In [ ]:
reduced_set = [
    candidate
    for candidate in range(1000, 10000)
    if BullsAndCowsGame.detect_bulls_and_cows(str(candidate), guess) == (2, 2)
]
print(reduced_set)

In [ ]:
facilitator_entropy = GameFacilitator(
    lambda: BullsAndCowsGame(),
    lambda: EntropyStrategy(feedback_table))

result = facilitator_entropy.play_games(1000, turn_frequency_log=50)
print(result)
print("All: ", result.total_games)
print("Won: ", result.won_games)
print("Average turns: ", result.average_turns)
print("Max: ", max(x.number_of_turns for x in result.all_games_results) )
print("Min: ", min(x.number_of_turns for x in result.all_games_results) )

# All:  1000
# Won:  1000
# Average turns:  5.728
# Max:  9
# Min:  3

####
# Using ALL possibilities for a guess, not just those that are still possible
# All:  100
# Won:  100
# Average turns: e

# Max:  10
# Min:  2


In [ ]:
plot_results_hist(result)

In [ ]:
my_secret = random.randint(1000, 10000)
my_secret = str(my_secret)

turns_counter = 0


solver = EntropyStrategy(feedback_table)
print("Secret is: ", my_secret)
while True:
    turns_counter += 1
    guess = solver.generate_guess()
    bulls, cows = BullsAndCowsGame.detect_bulls_and_cows(my_secret, guess)
    print(f"guess: {guess}. B&C: {(bulls, cows)}")
    solver.record_turn_feedback( bulls, cows )

    if bulls == 4:
        print(f"game won in: {turns_counter} turns")
        break

In [ ]:
facilitator_baseline = GameFacilitator(
    lambda: BullsAndCowsGame(),
    lambda: BaselineStrategy())

result = facilitator_baseline.play_games(1000)
print(result)
print("All: ", result.total_games)
print("Won: ", result.won_games)
print("Average turns: ", result.average_turns)
print("Max: ", max(x.number_of_turns for x in result.all_games_results) )
print("Min: ", min(x.number_of_turns for x in result.all_games_results) )

In [ ]:
plot_results_hist(result)

In [ ]:
# streammlit app flow
game = BullsAndCowsGame()
print(f"the secret is: {game.secret}")

strategy = EntropyStrategy(feedback_table)
while True:
    proposed_moves = strategy.generate_top_guesses()
    print(f"there is {strategy.calc_game_entropy()} bits of uncertainty in the current game")

    if game.game_state != GameState.IN_PROGRESS:
        print("You won")
        break

    print("You can consider: ")
    for (guess, average_entropy) in proposed_moves:
        print(f"- {guess} - {average_entropy}")

    guess = input("Your guess: ")
    if guess == "q":
        break
    strategy.set_guess(guess)
    (bulls, cows) = game.make_guess(guess)
    print(f"You got: bulls: {bulls} and cows: {cows}")
    strategy.record_turn_feedback(bulls, cows)
